# ComfyUI PromptDB — Kaggle T4 / TPU + Google Drive

同一份 Notebook 選擇 T4 或 TPU，保留 Google Drive 模型、工作流、PromptDB 提示詞、Cloudflare 公開介面及自動備份。

1. 在 Kaggle **Settings → Accelerator** 選擇 GPU T4 x2 或 TPU，並開啟 Internet；切換加速器會重新啟動工作階段。等 session 啟動後，執行下方選單儲存格，選擇相同模式，再依序執行後續儲存格。Run All 預設選 T4。
2. 在 **Add-ons → Secrets** 將既有 `RCLONE_CONFIG_TEXT` 附加到這份 Notebook；不得把憑證貼進儲存格。Drive 預設目錄為 `ComfyUI-PromptDB`。
3. 先從 Drive 同步 `models`、`workflows`、`user-db` 和 `logs`。Krea2 模型已存在且檢查通過時沿用；缺少才下載。保留 Drive 自訂模型以免破壞既有工作流。
4. 開啟後續儲存格輸出的 Cloudflare 連結，保持監控儲存格執行；原有提示庫和工作流位於 ComfyUI 的 PromptDB 介面。Cloudflare 連結具有公開性，請妥善保管。

TPU 模式及 Krea2 圖片生成仍需實機驗證；GPU T4 模式選單與 Kaggle 實際硬體必須一致。

若出現無法解析 github.com，請檢查 Kaggle 右側 Session options → Internet 是否為 On；若已開啟但仍失敗，重新啟動 Session 後再試。


In [ ]:
# 請先在 Kaggle Settings → Accelerator 選擇實際硬體，再在此選擇同一模式。
try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError as exc:
    raise RuntimeError('此互動選單需要 ipywidgets，請先開啟 Kaggle Internet。') from exc
accelerator_selector = widgets.ToggleButtons(
    options=[('GPU T4 x2', 'gpu'), ('TPU', 'tpu')], value='gpu', description='執行模式:'
)
display(accelerator_selector)
print('選單只切換 Notebook 軟體流程；加速器必須先在 Kaggle Settings 選擇。')


## 準備 Google Drive、模型、工作流、提示庫和 ComfyUI


In [ ]:
# 1. Options, repository checkout, rclone/Google Drive setup, and ComfyUI install
from pathlib import Path
import importlib.util
import subprocess
import socket
import tempfile
import time

STORE_MODELS_ON_DRIVE = True
DOWNLOAD_KREA2_MODELS = True
FORCE_VERIFY_MODELS = False
ENABLE_CLOUDFLARE_TUNNEL = True
BACKUP_OUTPUT = True
DRIVE_FOLDER = "ComfyUI-PromptDB"
RCLONE_SECRET_NAME = "RCLONE_CONFIG_TEXT"
BACKUP_INTERVAL_SECONDS = 120
REPO_BRANCH = "codex/kaggle-promptdb-t4-tpu"
REPO_URL = "https://github.com/JackChen6203/Google-Colab_Notebooks.git"
REPO = Path("/kaggle/temp/Google-Colab_Notebooks")

# Check the running session, not just the saved notebook metadata.
# DNS failure can mean Internet is off, network policy is blocking the session,
# or the Kaggle resolver is temporarily unavailable; never change DNS settings.
dns_ok = False
for attempt in range(1, 4):
    try:
        socket.getaddrinfo('github.com', 443, type=socket.SOCK_STREAM)
        dns_ok = True
        break
    except socket.gaierror:
        print(f'無法解析 github.com（第 {attempt}/3 次），稍後重試。', flush=True)
        if attempt < 3:
            time.sleep(3)
if not dns_ok:
    raise RuntimeError(
        'Kaggle 工作階段無法解析 github.com，因此尚未載入程式或連線 Google Drive。'
        '請在右側 Notebook / Session options 將 Internet 設為 On，然後重新執行此儲存格；'
        '若已是 On，請先 Stop Session 再 Start Session 後重試。'
        '若新工作階段仍失敗，請檢查帳號網路使用資格或向 Kaggle 回報 DNS 問題。'
    )

def git_with_retry(args, label):
    for attempt in range(1, 4):
        result = subprocess.run(args, capture_output=True, text=True, timeout=300)
        if result.returncode == 0:
            return
        print(f'{label} 失敗（第 {attempt}/3 次，git exit={result.returncode}）。', flush=True)
        if attempt < 3:
            time.sleep(5)
    raise RuntimeError(
        f'{label} 失敗。DNS 已可解析，但連線 GitHub 不成功；'
        '請確認 Kaggle Internet 已啟用，並稍後重試或檢查 Kaggle 網路狀態。'
    )

if REPO.exists():
    if not (REPO / '.git').is_dir():
        raise RuntimeError(f'{REPO} 已存在但不是 Git checkout；為保護現有資料，請使用新的 REPO 路徑。')
    git_with_retry(['git', '-C', str(REPO), 'fetch', '--depth', '1', 'origin', REPO_BRANCH], '更新 GitHub 程式')
    subprocess.run(['git', '-C', str(REPO), 'checkout', '-B', REPO_BRANCH, 'FETCH_HEAD'], check=True)
else:
    REPO.parent.mkdir(parents=True, exist_ok=True)
    # A failed clone lives only in an owned temporary directory; it cannot leave
    # an incomplete checkout at REPO or remove an existing user directory.
    with tempfile.TemporaryDirectory(prefix='promptdb-clone-', dir=REPO.parent) as tmp:
        for attempt in range(1, 4):
            attempt_checkout = Path(tmp) / f'checkout-{attempt}'
            result = subprocess.run(
                ['git', 'clone', '--depth', '1', '--single-branch', '--branch', REPO_BRANCH, REPO_URL, str(attempt_checkout)],
                capture_output=True, text=True, timeout=300,
            )
            if result.returncode == 0:
                attempt_checkout.rename(REPO)
                break
            print(f'下載 GitHub 程式失敗（第 {attempt}/3 次，git exit={result.returncode}）。', flush=True)
            if attempt < 3:
                time.sleep(5)
        else:
            raise RuntimeError('無法從 GitHub 下載程式。DNS 已可解析；請確認 Kaggle Internet 已啟用並稍後重試。')
print('GitHub 程式已就緒，繼續準備 Google Drive。', flush=True)

module_path = REPO / "promptdb_colab" / "kaggle_runtime.py"
spec = importlib.util.spec_from_file_location("promptdb_kaggle_runtime", module_path)
if spec is None or spec.loader is None:
    raise RuntimeError("Could not load PromptDB Kaggle runtime from the selected GitHub branch.")
runtime = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runtime)

context = runtime.bootstrap(
    REPO,
    drive_folder=DRIVE_FOLDER,
    rclone_secret_name=RCLONE_SECRET_NAME,
    store_models_on_drive=STORE_MODELS_ON_DRIVE,
    download_krea2_models=DOWNLOAD_KREA2_MODELS,
    force_verify_models=FORCE_VERIFY_MODELS,
    accelerator=accelerator_selector.value,
)

## 2. Start ComfyUI, Cloudflare Tunnel, and Drive backups

This cell waits for the local ComfyUI API, starts a Cloudflare Quick Tunnel, and checks the public UI and PromptDB API. The generated URL is temporary and public.

In [ ]:
# 2. Start ComfyUI, Cloudflare Tunnel, and automatic Drive backups
COMFYUI_URL = runtime.start_services(
    context,
    enable_cloudflare=ENABLE_CLOUDFLARE_TUNNEL,
    backup_output=BACKUP_OUTPUT,
    backup_interval=BACKUP_INTERVAL_SECONDS,
)

## 3. Monitor the ComfyUI queue

Keep this cell active while using ComfyUI. Queue changes and service diagnostics are saved under `gdrive:ComfyUI-PromptDB/logs`. Stop the cell normally to request a final Drive backup; Kaggle can still terminate a runtime without a final callback.

In [ ]:
# 3. Monitor the ComfyUI queue and keep the services active
runtime.monitor(context)